# NADI 2026 Subtask 2 — locating the validation→test gap

Three training runs score **89-90% on ADI20 validation** and **90.09% on ADI17 test**, and
**0.46-0.47 on the leaderboard**. This notebook exists to find out where the ~44 points go.

The test set (`UBC-NLP/NADI2026_subtask2_adi_test`) is **878 clips**, unlabeled, schema
`['audio', 'idx']`. Durations run roughly **0.4 s – 76 s** and are computed here from audio
headers (the `audioduration (s)` column shown on the HF dataset page is generated by the viewer,
not stored in the parquet). At n=878 a single clip is 0.11%, so:

| comparison | clips | std errors |
|---|---|---|
| 0.47 → 0.46 (v3 → v4) | 9 | 0.59 — **noise** |
| 0.47 → 0.51 (to 1st place) | 35 | 2.37 — real but marginal |

**Sections 1-4 are CPU-only** and run anywhere. **Sections 5-6 need the GPU box and the
checkpoint.** Every section prints a conclusion in words; if its inputs are missing it says what
it looked for and skips rather than half-running.

Order matters: §1 rules out the cheap catastrophic explanations before §5 spends GPU time on the
expensive one.

## 0 · Configuration

Edit the paths below, then run everything in order.

In [ ]:
import os, sys, io, json, math, glob, hashlib, warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

# ---- EDIT: the SAME cache the training runs used, or every corpus re-downloads.
# ---- On the vast.ai box that was /workspace/.hf_home (the training log prints it).
HF_CACHE_ROOT = os.environ.get("HF_HOME") or (
    "/workspace/.hf_home" if os.path.isdir("/workspace/.hf_home") else "")

def set_hf_cache(root):
    """Point BOTH caches at `root`, even if the libraries are already imported.

    datasets and huggingface_hub read their cache paths from the environment ONCE, at import
    time, into module-level constants. Setting os.environ afterwards is silently ignored -- which
    is exactly what happens when you re-run this notebook without restarting the kernel, and it
    looks identical to 'the cache is empty'. So patch the constants too.
    """
    if not root:
        print("HF cache: default (~/.cache/huggingface). Set HF_CACHE_ROOT to reuse an "
              "existing download.")
        return
    hub, dsets = os.path.join(root, "hub"), os.path.join(root, "datasets")
    os.environ.update(HF_HOME=root, HF_HUB_CACHE=hub, HF_DATASETS_CACHE=dsets)
    patched = []
    hc = sys.modules.get("huggingface_hub.constants")
    if hc is not None:
        for a, v in (("HF_HOME", root), ("HF_HUB_CACHE", hub), ("HUGGINGFACE_HUB_CACHE", hub)):
            if hasattr(hc, a):
                setattr(hc, a, v); patched.append(f"hub.{a}")
    dc = sys.modules.get("datasets.config")
    if dc is not None:
        for a, v in (("HF_DATASETS_CACHE", dsets), ("HF_CACHE_HOME", root), ("HF_HOME", root)):
            if hasattr(dc, a):
                setattr(dc, a, v); patched.append(f"datasets.{a}")
    print(f"HF cache root = {root}")
    print(f"  hub (downloads) {hub}   {'EXISTS' if os.path.isdir(hub) else 'MISSING -- will download'}")
    print(f"  datasets (arrow) {dsets} {'EXISTS' if os.path.isdir(dsets) else 'MISSING'}")
    if patched:
        print(f"  patched {len(patched)} already-imported constant(s) -- "
              "the libraries were loaded before this cell ran")
    if os.path.isdir(hub):
        repos = sorted(d for d in os.listdir(hub) if d.startswith("datasets--"))
        print(f"  cached dataset repos: {', '.join(r.replace('datasets--','') for r in repos) or '(none)'}")

set_hf_cache(HF_CACHE_ROOT)

# ---- EDIT: where your two submissions live ---------------------------------
SUBMISSIONS = {
    "v3": dict(logits="logits_v3.tsv", preds="predictions_v3.tsv", leaderboard=0.47),
    "v4": dict(logits="logits_v4.tsv", preds="predictions_v4.tsv", leaderboard=0.46),
}
CKPT = "best_v3_lm_vc_adi17_v4_full_c6_cohere-ar.pt"   # §5 only
OUT_DIR = "test_gap_out"
# Searched for any filename above, plus their immediate subdirectories.
SEARCH_DIRS = [".", "..", "/workspace", "/workspace/submissions",
               os.path.expanduser("~/Downloads")]
# -----------------------------------------------------------------------------

TEST_REPO   = "UBC-NLP/NADI2026_subtask2_adi_test"
TEST_SPLIT  = "train"          # the repo's only split is named 'train' despite being the test set
DUR_COL     = "audioduration (s)"
N_TEST      = 878              # from the dataset card; verified against the real load in §2

# Copied verbatim from cohere_train_v4.py:676 -- the submission column order depends on it.
COUNTRIES = ['MSA','BAH','TUN','ALG','EGY','IRA','JOR','KSA','KUW','LEB',
             'LIB','MAU','MOR','OMA','PAL','QAT','SUD','SYR','UAE','YEM']
labels2id = {k: i for i, k in enumerate(COUNTRIES)}
id2labels = {i: k for k, i in labels2id.items()}
TARGET_SR = 16000
MAX_AUDIO_SECONDS = 30         # MAX_AUDIO_SECONDS_COHERE, cohere_train_v4.py:2848

os.makedirs(OUT_DIR, exist_ok=True)

def find(name):
    """First existing match for `name` in SEARCH_DIRS or any immediate subdirectory of one."""
    if name and os.path.isabs(name) and os.path.exists(name):
        return name
    for d in SEARCH_DIRS:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    for d in SEARCH_DIRS:                       # one level down: ./submissions, ./ens_cache, ...
        if not os.path.isdir(d):
            continue
        for sub in sorted(glob.glob(os.path.join(d, "*", ""))):
            p = os.path.join(sub, name)
            if os.path.exists(p):
                return p
    return None


def header_durations(ds, label="corpus", cache=None):
    """Per-clip duration in seconds, read from audio file HEADERS -- no PCM decode.

    `Audio(decode=False)` hands back the raw {bytes, path} struct, and soundfile.info reads only
    the header, so this costs milliseconds per clip instead of a full resample. Results are
    cached to `cache` (a .npy path) because §2, §4 and §5 all join against the same vector.
    """
    if cache and os.path.exists(cache):
        d = np.load(cache)
        print(f"  {label}: {len(d)} durations from cache {cache}")
        return d
    import soundfile as sf
    # Fast path. cast_column triggers datasets' fingerprinting, which can fail on some
    # dill/Python combinations -- fall back to decoding rather than losing the whole section.
    fast = None
    try:
        from datasets import Audio
        fast = ds.cast_column("audio", Audio(decode=False))
    except Exception as e:
        print(f"  {label}: header-only read unavailable ({type(e).__name__}) -- decoding audio "
              "instead (slower, same answer)")
    src = fast if fast is not None else ds
    out, failed = [], 0
    for i in range(len(src)):
        a = src[i]["audio"]
        try:
            if fast is not None:
                info = sf.info(io.BytesIO(a["bytes"])) if a.get("bytes") else sf.info(a["path"])
                out.append(info.frames / info.samplerate)
            else:
                out.append(len(a["array"]) / int(a["sampling_rate"]))
        except Exception:
            out.append(np.nan); failed += 1
    d = np.asarray(out, dtype=np.float64)
    if failed:
        print(f"  {label}: {failed}/{len(ds)} header reads failed -> NaN")
    if cache:
        np.save(cache, d)
    return d

try:
    import matplotlib.pyplot as plt
    HAVE_PLT = True
except ImportError:
    HAVE_PLT = False
    print("matplotlib not available -- tables only, no plots.")

try:
    import torch
    HAVE_TORCH = True
    HAVE_GPU = torch.cuda.is_available()
except ImportError:
    HAVE_TORCH = HAVE_GPU = False

print(f"torch={'yes' if HAVE_TORCH else 'no'}  cuda={'yes' if HAVE_GPU else 'no'}  "
      f"matplotlib={'yes' if HAVE_PLT else 'no'}")
print(f"output -> {os.path.abspath(OUT_DIR)}")
print("\nfiles found:")
for tag, s in SUBMISSIONS.items():
    for k in ("logits", "preds"):
        p = find(s[k])
        print(f"  {tag}.{k:7s} {s[k]:28s} -> {p or 'NOT FOUND'}")
print(f"  ckpt        {CKPT:28s} -> {find(CKPT) or 'NOT FOUND (needed for §5 only)'}")

_stray = sorted({p for d in SEARCH_DIRS
                 for p in glob.glob(os.path.join(d, '*.tsv')) + glob.glob(os.path.join(d, '*', '*.tsv'))})
if _stray:
    print("\n.tsv files visible in SEARCH_DIRS (in case the names above are wrong):")
    for p in _stray[:20]:
        print(f"  {p}")

## 1 · Submission integrity

**Run this first.** Two failure modes here would explain the entire gap on their own, cost
seconds to check, and have never been checked:

1. **Append-mode double-write.** The official baseline's `write_logits` / `write_preds`
   (`NADI_2026__Streaming_Subset_Pipeline.ipynb`, cell 43) open with `"a"`. Running the
   submission cell twice silently doubles the file.
2. **Row-order misalignment.** The submission format has no `idx` column — *row order is the
   only join key*. A shuffled inference loader scrambles the submission while leaving every local
   metric perfect. That is the failure mode most consistent with "90% on two labeled sets, 46% on
   the leaderboard."

In [ ]:
def load_submission(tag):
    """Returns (logits ndarray (N,20) or None, preds ndarray (N,) or None, list-of-problems)."""
    s = SUBMISSIONS[tag]
    problems, logits, preds = [], None, None

    lp = find(s["logits"])
    if lp is None:
        problems.append(f"logits file {s['logits']!r} not found")
    else:
        rows, bad = [], 0
        for ln, line in enumerate(open(lp), 1):
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) != 20:
                bad += 1
                if bad <= 3:
                    problems.append(f"logits line {ln}: {len(parts)} fields, expected 20")
                continue
            rows.append([float(x) for x in parts])
        logits = np.asarray(rows, dtype=np.float64)
        if bad > 3:
            problems.append(f"...and {bad-3} more malformed logits lines")
        if not np.isfinite(logits).all():
            problems.append(f"{int((~np.isfinite(logits)).sum())} non-finite logit values")

    pp = find(s["preds"])
    if pp is None:
        problems.append(f"predictions file {s['preds']!r} not found")
    else:
        vals = [int(float(l.strip())) for l in open(pp) if l.strip()]
        preds = np.asarray(vals, dtype=np.int64)

    return logits, preds, problems


REPORT = {}
for tag in SUBMISSIONS:
    lg, pr, problems = load_submission(tag)
    REPORT[tag] = dict(logits=lg, preds=pr)
    print(f"=== {tag} " + "=" * 56)
    if lg is None and pr is None:
        print("  nothing loaded -- fix the paths in §0 and re-run.")
        for p in problems:
            print(f"  ! {p}")
        continue

    n_lg = 0 if lg is None else len(lg)
    n_pr = 0 if pr is None else len(pr)
    print(f"  logits rows      {n_lg}")
    print(f"  prediction rows  {n_pr}")

    # -- 1a. row count vs the test set
    for name, n in (("logits", n_lg), ("predictions", n_pr)):
        if n == 0:
            continue
        if n == N_TEST:
            print(f"  OK   {name}: exactly {N_TEST} rows")
        elif n % N_TEST == 0:
            k = n // N_TEST
            problems.append(
                f"{name} has {n} rows = {k}x{N_TEST}. This is the append-mode double-write: "
                f"write_{'logits' if name=='logits' else 'preds'}() opens with 'a', so the cell "
                f"was run {k} times. Only the first {N_TEST} rows were scored.")
        else:
            problems.append(f"{name} has {n} rows, expected {N_TEST} -- neither a match nor a "
                            "clean multiple. Row alignment with the test set is broken.")

    # -- 1b. logits/preds agreement
    if lg is not None and pr is not None and n_lg and n_pr:
        m = min(n_lg, n_pr)
        mism = int((lg[:m].argmax(1) != pr[:m]).sum())
        if mism == 0:
            print(f"  OK   argmax(logits) == predictions on all {m} shared rows")
        else:
            problems.append(f"argmax(logits) disagrees with predictions on {mism}/{m} rows "
                            f"({100*mism/m:.1f}%) -- the two files came from different passes.")

    # -- 1c. duplicate rows: a shuffled-then-appended file often repeats
    if lg is not None and n_lg:
        uniq = len({hashlib.md5(r.tobytes()).hexdigest() for r in np.ascontiguousarray(lg)})
        if uniq < n_lg:
            problems.append(f"only {uniq} unique logit rows out of {n_lg} "
                            f"({n_lg-uniq} duplicates)")
        else:
            print(f"  OK   all {n_lg} logit rows unique")

    if problems:
        print("\n  PROBLEMS:")
        for p in problems:
            print(f"  ! {p}")
    else:
        print("\n  clean: shape, count, and self-consistency all check out.")

In [ ]:
# -- 1d. What the leaderboard can and cannot resolve at n=878.
print(f"Binomial resolution at n={N_TEST}\n" + "-"*52)
rows = []
for tag, s in SUBMISSIONS.items():
    p = s["leaderboard"]
    se = math.sqrt(p*(1-p)/N_TEST)
    rows.append(dict(run=tag, score=p, correct=round(p*N_TEST), se=round(se, 4),
                     ci_lo=round(p-1.96*se, 3), ci_hi=round(p+1.96*se, 3)))
first = 0.51
se1 = math.sqrt(first*(1-first)/N_TEST)
rows.append(dict(run="1st place", score=first, correct=round(first*N_TEST), se=round(se1, 4),
                 ci_lo=round(first-1.96*se1, 3), ci_hi=round(first+1.96*se1, 3)))
display(pd.DataFrame(rows).set_index("run"))

scores = [s["leaderboard"] for s in SUBMISSIONS.values()]
if len(scores) == 2:
    d = abs(scores[0]-scores[1]); pbar = sum(scores)/2
    z = d/math.sqrt(pbar*(1-pbar)/N_TEST) if d else 0.0
    print(f"\nyour two submissions differ by {round(d*N_TEST)} clips = {z:.2f} se "
          f"-> {'INDISTINGUISHABLE' if z < 1.96 else 'significant'}")
gap = first - max(scores)
zg = gap/math.sqrt(0.49*0.51/N_TEST)
print(f"gap to 1st place: {round(gap*N_TEST)} clips = {zg:.2f} se "
      f"-> {'not resolvable' if zg < 1.96 else 'real, but a small margin'}")
print("\nCONCLUSION: any future change worth submitting has to be worth more than ~30 clips.\n"
      "Differences below ~3.3 points cannot be measured on this leaderboard.")

## 2 · Test-set profile

The dataset has no duration column — the one on the HF page is computed by the viewer. Durations
are read from the audio **file headers** (`soundfile.info`, no PCM decode), which for 878 clips
takes seconds, and are cached to `test_durations.npy` so later cells are instant.

Two thresholds matter:
- **under ~2-3 s**: dialect ID becomes very hard; there simply isn't enough phonetic evidence.
- **over 30 s**: `MAX_AUDIO_SECONDS_COHERE = 30` truncates at inference, so anything longer is
  partially discarded.

In [ ]:
from datasets import load_dataset

try:
    _t = load_dataset(TEST_REPO, split=TEST_SPLIT)
    test_cols = list(_t.column_names)
    print(f"loaded {TEST_REPO}[{TEST_SPLIT}]: {len(_t)} rows, columns {test_cols}")
    if len(_t) != N_TEST:
        print(f"  NOTE: {len(_t)} rows, not the {N_TEST} the dataset card showed. "
              "Using the real count from here on.")
        N_TEST = len(_t)
    # The HF dataset VIEWER displays an 'audioduration (s)' column, but it is computed by the
    # viewer for display -- it is not in the parquet. Use it if a future revision adds it for
    # real; otherwise read durations from the audio headers (878 clips, a few seconds).
    dur_col = next((c for c in test_cols if "dur" in c.lower()), None)
    if dur_col:
        print(f"  using duration column {dur_col!r}")
        test_dur = np.asarray(_t[dur_col], dtype=np.float64)
    else:
        print(f"  no duration column in {test_cols} (the viewer computes that one) -- "
              "reading headers instead")
        test_dur = header_durations(_t, label="test",
                                    cache=os.path.join(OUT_DIR, "test_durations.npy"))
    test_idx = list(_t["idx"]) if "idx" in test_cols else list(range(len(_t)))
    _nan = int(np.isnan(test_dur).sum())
    if _nan:
        print(f"  {_nan} clips have no readable duration; they are excluded from duration joins")
    print(f"  durations: min {np.nanmin(test_dur):.2f}s  median {np.nanmedian(test_dur):.2f}s  "
          f"max {np.nanmax(test_dur):.2f}s  total {np.nansum(test_dur)/3600:.2f}h")
    has_labels = any(c in test_cols for c in ("dialect", "label", "country"))
    print(f"labels present in test set: {has_labels}"
          + ("" if not has_labels else "  <-- unexpected; §4/§5 could use them directly"))
    TEST_OK = True
except Exception as e:
    print(f"could not load the test set: {type(e).__name__}: {e}")
    print("§2-§4 duration joins will be skipped. Check HF auth if the repo is gated.")
    test_dur, test_idx, TEST_OK = None, None, False

In [ ]:
def dur_summary(d, name):
    d = np.asarray(d, dtype=float)
    q = [1, 5, 10, 25, 50, 75, 90, 99]
    row = {f"p{x}": round(float(np.nanpercentile(d, x)), 2) for x in q}
    row.update(n=int(np.isfinite(d).sum()), min=round(float(np.nanmin(d)), 2),
               max=round(float(np.nanmax(d)), 2), mean=round(float(np.nanmean(d)), 2),
               total_h=round(float(np.nansum(d)/3600), 2))
    return pd.Series(row, name=name)

if TEST_OK:
    display(dur_summary(test_dur, "test").to_frame().T)

    print("share of the test set by duration band")
    print("-"*52)
    bands = [(0,1),(1,2),(2,3),(3,5),(5,8),(8,12),(12,20),(20,30),(30,1e9)]
    cum = 0
    for lo, hi in bands:
        m = (test_dur >= lo) & (test_dur < hi)
        pct = 100*m.sum()/len(test_dur); cum += pct
        lbl = f"{lo:>5.0f}-{hi:<5.0f}s" if hi < 1e9 else f"  >{lo:.0f}s (truncated)"
        print(f"  {lbl} {m.sum():5d} clips  {pct:5.1f}%   cum {cum:5.1f}%")

    short = float((test_dur < 3).mean()*100)
    trunc = float((test_dur > MAX_AUDIO_SECONDS).mean()*100)
    print(f"\n  under 3 s: {short:.1f}%   over {MAX_AUDIO_SECONDS} s (truncated): {trunc:.1f}%")

    est_mb = np.nansum(test_dur) * TARGET_SR * 2 / 1e6
    print(f"  sanity: {np.nansum(test_dur)/3600:.2f} h at 16 kHz 16-bit mono = {est_mb:.0f} MB "
          f"(card says 194 MB)")

    if HAVE_PLT:
        _d = test_dur[np.isfinite(test_dur)]
        fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
        ax[0].hist(_d, bins=60, color="#c0392b")
        ax[0].axvline(3, ls="--", c="k", lw=1); ax[0].axvline(MAX_AUDIO_SECONDS, ls=":", c="k", lw=1)
        ax[0].set_xlabel("duration (s)"); ax[0].set_ylabel("clips"); ax[0].set_title("test durations")
        ax[1].hist(np.log10(np.clip(_d, 1e-3, None)), bins=60, color="#c0392b")
        ax[1].set_xlabel("log10 duration (s)"); ax[1].set_title("test durations (log)")
        plt.tight_layout(); plt.show()

## 3 · Reference-set profile

The same statistics for **ADI20 validation** and **Casablanca holdout**, so the test distribution
has something to be compared against.

Durations are read from **file headers only** (`Audio(decode=False)` + `soundfile.info`), which
avoids decoding 10,806 clips.

**If this starts downloading, stop it.** It means `datasets` resolved its cache before §0 ran —
restart the kernel and run §0 first. §3 loads validation by resolving *only* the validation
parquet files and passing them explicitly, so sibling splits can never be pulled in; the cell
prints the resolved cache paths and the file count so you can see what it is about to fetch.
Set `LOAD_REFERENCE_CORPORA = False` to skip this section entirely — §1, §2 and §4 don't need it.

In [ ]:
# SET THIS FALSE to skip the reference corpora entirely. They are only needed for the §3
# comparison and the §5 gate -- §1, §2 and §4 do not touch them.
LOAD_REFERENCE_CORPORA = True

# If HF_HOME above does not match the cache the training runs used, these re-download tens of
# GB. Check the printed HF_HOME before running this cell.
REF_DUR = {}
_val = None
if not LOAD_REFERENCE_CORPORA:
    print("LOAD_REFERENCE_CORPORA is False -- skipping §3 and the §5 gate.")
else:
    import datasets.config as _dc, huggingface_hub.constants as _hc
    print(f"resolved caches: datasets={_dc.HF_DATASETS_CACHE}\n"
          f"                 hub     ={_hc.HF_HUB_CACHE}")
    print("If those are not under HF_CACHE_ROOT, restart the kernel and run §0 FIRST.\n")

    def load_split_files_only(repo, split):
        """Load exactly one split by resolving its parquet files and passing them explicitly.

        load_dataset(repo, split=...) asks `datasets` to resolve the repo's splits for itself,
        and on a layout it cannot map cleanly it materialises every split before selecting one.
        This is the same fix that stopped cohere_train_v4.py pulling ADI17's 260 GB train split
        to hand back 3 GB of test.
        """
        from huggingface_hub import HfFileSystem
        fs = HfFileSystem()
        root = f"datasets/{repo}"
        files = []
        for pat in (f"{root}/**/{split}-*.parquet",
                    f"{root}/**/{split}/*.parquet",
                    f"{root}/{split}/*.parquet"):
            files = sorted(fs.glob(pat))
            if files:
                break
        if not files:
            files = sorted(f for f in fs.glob(f"{root}/**/*.parquet")
                           if f"/{split}/" in f or f"/{split}-" in f)
        if not files:
            raise FileNotFoundError(f"no {split!r} parquet files under {root}")
        print(f"  {repo}[{split}]: {len(files)} parquet file(s) "
              f"(repo has {len(fs.glob(f'{root}/**/*.parquet'))} total)")
        d = load_dataset("parquet", data_files=["hf://" + f for f in files], split="train")
        if "audio" in d.column_names:
            from datasets import Audio
            if not isinstance(d.features.get("audio"), Audio):
                d = d.cast_column("audio", Audio(sampling_rate=TARGET_SR))
        return d

    try:
        _val = load_split_files_only("UBC-NLP/NADI_2026_ADI20_micro", "validation")
        REF_DUR["ADI20 val"] = header_durations(
            _val, label="ADI20 val", cache=os.path.join(OUT_DIR, "adi20_val_durations.npy"))
        print(f"ADI20 val: {len(REF_DUR['ADI20 val'])} durations")
    except Exception as e:
        print(f"ADI20 val unavailable ({type(e).__name__}: {e}) -- skipping")

if LOAD_REFERENCE_CORPORA:
    try:
        from datasets import concatenate_datasets
        CASA = {"Algeria":"ALG","Egypt":"EGY","Jordan":"JOR","Mauritania":"MAU",
                "Morocco":"MOR","Palestine":"PAL","UAE":"UAE","Yemen":"YEM"}
        parts = []
        for cfg in CASA:
            try:
                d = load_dataset("UBC-NLP/Casablanca", cfg)
                for sp in ("validation", "test"):
                    if sp in d:
                        parts.append(d[sp].select_columns(["audio"]))
            except Exception:
                pass
        if parts:
            REF_DUR["Casablanca"] = header_durations(
                concatenate_datasets(parts), label="Casablanca",
                cache=os.path.join(OUT_DIR, "casablanca_durations.npy"))
            print(f"Casablanca: {len(REF_DUR['Casablanca'])} durations")
    except Exception as e:
        print(f"Casablanca unavailable ({type(e).__name__}) -- skipping")

In [ ]:
if TEST_OK and REF_DUR:
    tbl = [dur_summary(test_dur, "TEST")] + [dur_summary(d, k) for k, d in REF_DUR.items()]
    display(pd.DataFrame(tbl))

    _td = test_dur[np.isfinite(test_dur)]
    try:
        from scipy.stats import ks_2samp
        print("\nKolmogorov-Smirnov vs the test duration distribution")
        print("-"*52)
        for k, d in REF_DUR.items():
            ks = ks_2samp(_td, d[np.isfinite(d)])
            print(f"  {k:14s} D={ks.statistic:.3f}  p={ks.pvalue:.2e}"
                  f"   {'LARGE shift' if ks.statistic > 0.3 else 'moderate' if ks.statistic > 0.15 else 'small'}")
    except ImportError:
        print("\nscipy unavailable -- comparing percentiles instead:")
        for k, d in REF_DUR.items():
            print(f"  {k:14s} p10/p50/p90 = "
                  f"{np.nanpercentile(d,10):.1f}/{np.nanpercentile(d,50):.1f}/{np.nanpercentile(d,90):.1f}"
                  f"   vs test "
                  f"{np.nanpercentile(_td,10):.1f}/{np.nanpercentile(_td,50):.1f}/"
                  f"{np.nanpercentile(_td,90):.1f}")

    print("\nshare under 3 s")
    print(f"  TEST           {100*(_td<3).mean():5.1f}%")
    for k, d in REF_DUR.items():
        print(f"  {k:14s} {100*(d[np.isfinite(d)]<3).mean():5.1f}%")

    if HAVE_PLT:
        plt.figure(figsize=(9, 3.6))
        bins = np.linspace(0, 40, 80)
        plt.hist(np.clip(_td, 0, 40), bins=bins, density=True, alpha=.55,
                 label=f"TEST (n={len(_td)})", color="#c0392b")
        for k, d in REF_DUR.items():
            plt.hist(np.clip(d[np.isfinite(d)], 0, 40), bins=bins, density=True, histtype="step", lw=2, label=k)
        plt.axvline(3, ls="--", c="k", lw=1); plt.axvline(MAX_AUDIO_SECONDS, ls=":", c="k", lw=1)
        plt.xlabel("duration (s)"); plt.ylabel("density"); plt.legend()
        plt.title("test vs labeled reference sets"); plt.tight_layout(); plt.show()
elif TEST_OK:
    print("No reference corpora loaded -- run this on the GPU box where they're cached.")

## 4 · Prediction-distribution analysis

The test set is unlabeled, so accuracy can't be computed here. But two things stand in for it:

- **Confidence** (max softmax / entropy) — low where the model is struggling.
- **Inter-model agreement** between the v3 and v4 submissions — two independently-trained models
  agreeing is weak evidence of correctness; disagreeing is strong evidence at least one is wrong.

Joining both against the per-clip durations from §2 localises *where* the model fails without
ever needing labels.

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

USABLE = {t: r for t, r in REPORT.items() if r["logits"] is not None and len(r["logits"])}
for tag, r in USABLE.items():
    if len(r["logits"]) > N_TEST:
        print(f"NOTE: {tag} has {len(r['logits'])} logit rows; using the first {N_TEST}. "
              "This mirrors what the scorer saw, but see the §1 warning -- fix the file.")
    lg = r["logits"][:N_TEST]
    p = softmax(lg)
    r["prob"], r["pred"] = p, lg.argmax(1)
    r["conf"] = p.max(1)
    r["entropy"] = -(p * np.log(np.clip(p, 1e-12, None))).sum(1)

if not USABLE:
    print("no usable logits -- fix §0 paths.")
else:
    print(f"predicted-class distribution ({N_TEST} clips, uniform would be {100/20:.1f}%)")
    dist = pd.DataFrame({
        tag: pd.Series(r["pred"]).value_counts().reindex(range(20), fill_value=0).values
        for tag, r in USABLE.items()
    }, index=COUNTRIES)
    for tag in USABLE:
        dist[f"{tag} %"] = (100*dist[tag]/dist[tag].sum()).round(1)
    display(dist.sort_values(dist.columns[0], ascending=False))

    for tag, r in USABLE.items():
        c = pd.Series(r["pred"]).value_counts(normalize=True)
        top = c.head(3)
        print(f"\n{tag}: top-3 predicted = " +
              ", ".join(f"{COUNTRIES[i]} {100*v:.1f}%" for i, v in top.items()) +
              f"   (uniform 5.0%)")
        n_never = int((dist[tag] == 0).sum())
        print(f"     {n_never}/20 countries never predicted; "
              f"max-class share {100*c.iloc[0]:.1f}%")
        # Chi-square vs uniform: is the skew real, or just 878 samples?
        obs = dist[tag].values.astype(float); exp = obs.sum()/20
        chi2 = float(((obs-exp)**2/exp).sum())
        print(f"     chi2 vs uniform = {chi2:.0f} (df=19; >30 means clearly non-uniform)")

In [ ]:
if len(USABLE) >= 2:
    tags = list(USABLE)[:2]
    a, b = USABLE[tags[0]], USABLE[tags[1]]
    m = min(len(a["pred"]), len(b["pred"]))
    agree = a["pred"][:m] == b["pred"][:m]
    rate = 100*agree.mean()
    print(f"{tags[0]} vs {tags[1]}: agree on {int(agree.sum())}/{m} clips ({rate:.1f}%)")
    print(f"  both score ~0.46-0.47, so if agreement were perfect they would be the same model.")
    if rate > 90:
        print("  VERDICT: highly correlated -- an ensemble has little headroom.")
    elif rate > 75:
        print("  VERDICT: moderately decorrelated -- an ensemble is worth trying (§6).")
    else:
        print("  VERDICT: strongly decorrelated -- an ensemble should give a real gain (§6).")

    # Upper bound: if every disagreement were resolved correctly, what could an oracle reach?
    both_wrong_floor = 1 - (SUBMISSIONS[tags[0]]["leaderboard"] + SUBMISSIONS[tags[1]]["leaderboard"])
    print(f"\n  disagreements: {int((~agree).sum())} clips ({100*(~agree).mean():.1f}%). "
          f"A per-clip oracle choosing the better model could reach at most "
          f"{min(1.0, SUBMISSIONS[tags[0]]['leaderboard'] + (~agree).mean()):.2f} "
          "(a loose bound -- it assumes every disagreement resolves in your favour).")

    print("\n  where they disagree, who is more confident?")
    dis = ~agree
    if dis.sum():
        ca, cb = a["conf"][:m][dis], b["conf"][:m][dis]
        print(f"    {tags[0]} mean conf {ca.mean():.3f} | {tags[1]} mean conf {cb.mean():.3f}")
        print(f"    {tags[0]} more confident on {100*(ca>cb).mean():.1f}% of disagreements")

In [ ]:
# The key join: does the model fail on SHORT clips?
if TEST_OK and USABLE:
    n = min(N_TEST, len(test_dur), min(len(r["pred"]) for r in USABLE.values()))
    d = test_dur[:n]
    if np.isnan(d).any():
        print(f"  {int(np.isnan(d).sum())} clips have no duration; excluded from these bands")
    edges = [0, 1, 2, 3, 5, 8, 12, 20, 30, 1e9]
    rows = []
    tags = list(USABLE)
    for lo, hi in zip(edges[:-1], edges[1:]):
        msk = (d >= lo) & (d < hi)
        if msk.sum() == 0:
            continue
        row = {"band": f"{lo:g}-{hi:g}s" if hi < 1e9 else f">{lo:g}s", "n": int(msk.sum())}
        for t in tags:
            row[f"conf {t}"] = round(float(USABLE[t]["conf"][:n][msk].mean()), 3)
            row[f"H {t}"] = round(float(USABLE[t]["entropy"][:n][msk].mean()), 2)
        if len(tags) >= 2:
            ag = USABLE[tags[0]]["pred"][:n][msk] == USABLE[tags[1]]["pred"][:n][msk]
            row["agree %"] = round(100*float(ag.mean()), 1)
        rows.append(row)
    band = pd.DataFrame(rows).set_index("band")
    display(band)

    print("READ THIS TABLE AS: confidence and agreement are proxies for accuracy.")
    if "agree %" in band.columns and len(band) > 2:
        lo_bands = band[band.index.str.startswith(("0-", "1-", "2-"))]
        hi_bands = band[~band.index.str.startswith(("0-", "1-", "2-"))]
        if len(lo_bands) and len(hi_bands):
            da = lo_bands["agree %"].mean() - hi_bands["agree %"].mean()
            print(f"\n  agreement under 3 s vs over 3 s: {lo_bands['agree %'].mean():.1f}% vs "
                  f"{hi_bands['agree %'].mean():.1f}%  (delta {da:+.1f} pts)")
            share_short = 100*(d < 3).mean()
            if da < -10:
                print(f"  VERDICT: short clips are a distinct failure mode, and they are "
                      f"{share_short:.1f}% of the test set. §5 quantifies the cost.")
            else:
                print(f"  VERDICT: no strong short-clip effect. The gap is more likely DOMAIN "
                      f"than duration -- §5 will confirm.")

    if HAVE_PLT and len(band) > 2:
        fig, ax = plt.subplots(figsize=(9, 3.4))
        x = range(len(band))
        for t in tags:
            ax.plot(x, band[f"conf {t}"], marker="o", label=f"confidence {t}")
        ax.set_xticks(list(x)); ax.set_xticklabels(band.index, rotation=45)
        ax.set_ylabel("mean max-softmax"); ax.legend(loc="lower right")
        if "agree %" in band.columns:
            ax2 = ax.twinx()
            ax2.plot(x, band["agree %"], marker="s", ls="--", c="gray", label="agreement %")
            ax2.set_ylabel("v3/v4 agreement %")
        ax.set_title("confidence and agreement vs clip duration")
        plt.tight_layout(); plt.show()

## 5 · The decisive experiment — GPU required

Everything above is circumstantial. This converts it into a number.

Take **labeled** ADI20 validation, re-evaluate the checkpoint two ways, and compare:

1. **native durations** → must reproduce the published **89.32%**. This is a correctness gate on
   the model rebuilt below; if it fails, nothing downstream can be trusted.
2. **durations resampled from the test set's empirical distribution** → if accuracy collapses
   toward 50%, the gap is duration and the fix is inference-side. If it barely moves, the gap is
   domain and cropping won't help.

`cohere_train_v4.py` cannot be imported — `ARGS = get_args()` runs at line 643 and the module
loads datasets at import time — so the inference path is reproduced here from
`DialectID` (line 2325), `_extract_features` (2854) and `_derive_frame_mask` (2445).

In [ ]:
if not HAVE_GPU:
    print("No CUDA device. §5-§6 model sections are skipped -- run this notebook on the GPU box.")
else:
    import torch.nn as nn
    import inspect
    from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
    device = torch.device("cuda")
    HF_ID = "CohereLabs/cohere-transcribe-arabic-07-2026"

    def _encoder_layer_list(encoder):
        best = None
        for name, mod in encoder.named_modules():
            if isinstance(mod, nn.ModuleList) and len(mod) >= 4:
                if best is None or len(mod) > len(best):
                    best = mod
        return best if best is not None else []

    def _resolve_mask_kwarg(encoder):
        try:
            sig = inspect.signature(encoder.forward)
        except (TypeError, ValueError):
            return None
        for nm in ("attention_mask","padding_mask","input_lengths","lengths","feature_lengths"):
            if nm in sig.parameters:
                return nm
        return None

    class DialectID(nn.Module):
        """Inference-only copy of cohere_train_v4.py's DialectID (mean pool + layer-mix)."""
        def __init__(self, num_labels=20, layer_mix=True):
            super().__init__()
            full = AutoModelForSpeechSeq2Seq.from_pretrained(HF_ID, trust_remote_code=True).float()
            self.encoder = full.get_encoder()
            cfg = self.encoder.config
            hidden = getattr(cfg, "d_model", None) or cfg.hidden_size
            self._mask_kwarg = _resolve_mask_kwarg(self.encoder)
            self._layer_mix = layer_mix
            n_layers = len(_encoder_layer_list(self.encoder))
            self.layer_weights = nn.Parameter(torch.zeros(n_layers + 1)) if layer_mix else None
            self.classifier = nn.Linear(hidden, num_labels)
            print(f"  encoder mask kwarg: {self._mask_kwarg!r} | {n_layers} layers | hidden {hidden}")

        def _derive_frame_mask(self, out, h, input_mask):
            T = h.shape[1]
            for attr in ("output_lengths","encoder_out_lens","lengths","output_length"):
                lens = getattr(out, attr, None)
                if lens is not None:
                    lens = lens.to(h.device)
                    return torch.arange(T, device=h.device).unsqueeze(0) < lens.unsqueeze(1)
            if input_mask is not None:
                if input_mask.shape[1] == T:
                    return input_mask.bool()
                m = input_mask.float().unsqueeze(1)
                return torch.nn.functional.interpolate(m, size=T, mode="nearest").squeeze(1).bool()
            return None

        def _mix(self, out, h_last):
            hs = list(getattr(out, "hidden_states", None) or [])
            if len(hs) < 2:
                raise RuntimeError("encoder did not populate hidden_states")
            if hs[-1] is not h_last and h_last is not None:
                hs.append(h_last)
            w = self.layer_weights
            if w.numel() != len(hs):
                with torch.no_grad():
                    new = torch.zeros(len(hs), device=w.device, dtype=w.dtype)
                    new[:min(len(hs), w.numel())] = w[:min(len(hs), w.numel())]
                    self.layer_weights.data = new
            weights = torch.softmax(self.layer_weights, 0).to(hs[-1].dtype)
            return torch.stack(hs, 0).mul(weights.view(-1,1,1,1)).sum(0)

        def forward(self, input_features=None, attention_mask=None):
            kw = {}
            if self._mask_kwarg and attention_mask is not None:
                kw[self._mask_kwarg] = attention_mask
            if self._layer_mix:
                kw["output_hidden_states"] = True
            out = self.encoder(input_features, **kw)
            h = out.last_hidden_state if hasattr(out, "last_hidden_state") else out[0]
            if self._layer_mix:
                h = self._mix(out, h)
            mask = self._derive_frame_mask(out, h, attention_mask)
            if mask is None:
                pooled = h.mean(1)
            else:
                m = mask.float().unsqueeze(-1)
                pooled = (h*m).sum(1) / m.sum(1).clamp(min=1.0)
            return self.classifier(pooled)

    def _extract_features(fe, wavs):
        try:
            raw = fe(wavs, sampling_rate=TARGET_SR, return_tensors="pt", padding=True)
        except TypeError:
            raw = fe(wavs, sampling_rate=TARGET_SR, return_tensors="pt")
        key = "input_features" if "input_features" in raw else "input_values"
        mask = None
        for mk in ("attention_mask","input_features_mask","feature_attention_mask"):
            if mk in raw:
                mask = raw[mk]; break
        return raw[key], mask

    def set_bn_eval(module):
        for m in module.modules():
            if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d, nn.SyncBatchNorm)):
                m.eval()

    print("model classes defined.")

In [ ]:
if HAVE_GPU:
    ckpt_path = find(CKPT)
    assert ckpt_path, f"checkpoint {CKPT} not found in {SEARCH_DIRS}"
    fe = AutoProcessor.from_pretrained(HF_ID, trust_remote_code=True)
    model = DialectID(layer_mix=True).to(device)
    state = torch.load(ckpt_path, map_location=device)
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    # Size layer_weights to the checkpoint before loading, or the load silently skips it.
    if "layer_weights" in state and model.layer_weights is not None:
        if state["layer_weights"].shape != model.layer_weights.shape:
            model.layer_weights.data = torch.zeros_like(state["layer_weights"]).to(device)
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"loaded {ckpt_path}")
    print(f"  missing {len(missing)} | unexpected {len(unexpected)}")
    for k in list(missing)[:5]:    print(f"    missing:    {k}")
    for k in list(unexpected)[:5]: print(f"    unexpected: {k}")
    assert len(missing) < 10, ("too many missing keys -- the rebuilt model does not match the "
                               "checkpoint architecture. Fix before trusting any number below.")
    model.eval(); set_bn_eval(model)
    w = torch.softmax(model.layer_weights.detach().float(), 0).cpu().numpy()
    print(f"  layer-mix: peak layer {int(w.argmax())}/{len(w)-1}, max weight {w.max():.4f}")

In [ ]:
if HAVE_GPU:
    @torch.no_grad()
    def evaluate(ds, label_col="dialect", label_map=None, crop_fn=None, bs=16, desc=""):
        """Returns (preds, labels, durations). crop_fn(wav)->wav is applied per clip."""
        label_map = label_map or labels2id
        P, Y, D = [], [], []
        buf_w, buf_y, buf_d = [], [], []

        def flush():
            if not buf_w: return
            feats, mask = _extract_features(fe, buf_w)
            feats = feats.to(device)
            mask_d = mask.to(device) if mask is not None else None
            with torch.autocast("cuda", dtype=torch.bfloat16):
                lg = model(input_features=feats, attention_mask=mask_d)
            P.extend(lg.float().argmax(-1).cpu().tolist()); Y.extend(buf_y); D.extend(buf_d)
            buf_w.clear(); buf_y.clear(); buf_d.clear()

        for i in range(len(ds)):
            s = ds[i]
            try:
                arr, sr = s["audio"]["array"], int(s["audio"]["sampling_rate"])
            except Exception:
                continue
            w = torch.as_tensor(arr, dtype=torch.float32)
            if w.ndim > 1: w = w.mean(0)
            if sr != TARGET_SR:
                w = torch.from_numpy(np.interp(
                    np.linspace(0, len(w)-1, int(len(w)*TARGET_SR/sr)),
                    np.arange(len(w)), w.numpy()).astype(np.float32))
            dur = len(w)/TARGET_SR
            if crop_fn is not None:
                w = crop_fn(w)
            w = w[:MAX_AUDIO_SECONDS*TARGET_SR]
            if w.numel() < 400:
                continue
            lab = label_map.get(s[label_col]) if label_col in s else None
            if lab is None:
                continue
            buf_w.append(w.numpy()); buf_y.append(lab); buf_d.append(dur)
            if len(buf_w) == bs:
                flush()
        flush()
        if desc:
            acc = 100*np.mean(np.array(P) == np.array(Y))
            print(f"  {desc}: {acc:.2f}%  (n={len(P)})")
        return np.array(P), np.array(Y), np.array(D)

if HAVE_GPU and globals().get("_val") is None:
    print("ADI20 val was not loaded in §3 -- run that cell first, or point it at the cached copy.\n"
          "Without labeled data there is no gate and no §5 result.")
elif HAVE_GPU:
    print("GATE: reproducing the published ADI20 val number (must be 89.32%)")
    p0, y0, d0 = evaluate(_val, desc="ADI20 val, native durations")
    acc0 = 100*np.mean(p0 == y0)
    if abs(acc0 - 89.32) < 0.5:
        print("  OK -- the rebuilt inference path matches cohere_train_v4.py. Proceed.")
    else:
        print(f"  MISMATCH: got {acc0:.2f}, expected 89.32. The rebuilt forward path has "
              "diverged. Do NOT trust anything below until this is resolved.")

In [ ]:
if not (HAVE_GPU and TEST_OK and "acc0" in globals()):
    print("Skipping: needs a GPU, the test durations from §2, and the §5 gate to have run.")
else:
    rng = np.random.default_rng(42)

    def make_crop_fn(target_durations):
        pool = np.asarray(target_durations)
        def crop(w):
            d = float(rng.choice(pool))
            n = max(400, int(d*TARGET_SR))
            if len(w) <= n:
                return w
            st = int(rng.integers(0, len(w)-n))
            return w[st:st+n]
        return crop

    print("ADI20 val re-evaluated under the TEST set's duration distribution")
    p1, y1, _ = evaluate(_val, crop_fn=make_crop_fn(test_dur), desc="duration-matched")
    acc1 = 100*np.mean(p1 == y1)
    drop = acc0 - acc1
    print(f"\n  native {acc0:.2f}%  ->  test-duration-matched {acc1:.2f}%   (drop {drop:.2f} pts)")
    lead = max(s['leaderboard'] for s in SUBMISSIONS.values())*100
    print(f"  leaderboard is {lead:.0f}%, i.e. {acc0-lead:.0f} pts below native val.")
    print(f"  duration accounts for {100*drop/max(1e-9, acc0-lead):.0f}% of that gap.")
    if drop > 20:
        print("\n  VERDICT: DURATION is a major driver. Fix at inference: short-clip TTA, and\n"
              "  train with crops drawn from the test duration distribution.")
    elif drop > 8:
        print("\n  VERDICT: duration is a real but partial driver. Domain shift carries the rest.")
    else:
        print("\n  VERDICT: duration is NOT the explanation. The gap is DOMAIN -- different\n"
              "  recording conditions or speech style. More/different data, not better cropping.")

    # accuracy vs native duration, directly comparable to §4's confidence curve
    rows = []
    for lo, hi in zip([0,1,2,3,5,8,12,20,30],[1,2,3,5,8,12,20,30,1e9]):
        m = (d0 >= lo) & (d0 < hi)
        if m.sum() < 10: continue
        rows.append(dict(band=f"{lo:g}-{hi:g}s" if hi < 1e9 else f">{lo:g}s",
                         n=int(m.sum()), acc=round(100*float((p0[m]==y0[m]).mean()), 1)))
    if rows:
        print("\naccuracy vs native clip duration (ADI20 val, labeled):")
        display(pd.DataFrame(rows).set_index("band"))
        print("Compare this shape against the confidence/agreement curve in §4 -- if they match,\n"
              "the unlabeled proxies are trustworthy and can be used on the test set directly.")

## 6 · Remedies, scored before submitting

Only changes that can be validated on labeled data first. Each writes a candidate submission in
the official format — 20 tab-separated floats per line, one line per test clip, **`"w"` not
`"a"`**, original row order preserved.

In [ ]:
def write_submission(logits, stem):
    """Official format. Opens with 'w' -- the baseline's 'a' is what doubles files."""
    lp = os.path.join(OUT_DIR, f"logits_{stem}.tsv")
    pp = os.path.join(OUT_DIR, f"predictions_{stem}.tsv")
    with open(lp, "w") as f:
        for row in logits:
            f.write("\t".join(str(float(v)) for v in row) + "\n")
    with open(pp, "w") as f:
        for i in logits.argmax(1):
            f.write(f"{int(i)}\n")
    # round-trip check
    back = np.array([[float(x) for x in l.split("\t")] for l in open(lp) if l.strip()])
    preds_back = np.array([int(l) for l in open(pp) if l.strip()])
    assert back.shape == logits.shape, f"round-trip shape {back.shape} != {logits.shape}"
    assert (back.argmax(1) == preds_back).all(), "round-trip argmax mismatch"
    assert np.isfinite(back).all(), "round-trip produced non-finite values"
    print(f"  wrote {lp} and {pp}  ({back.shape[0]} rows x {back.shape[1]}) -- round-trip OK")
    return lp, pp


# --- remedy A: prior correction ------------------------------------------------
# Subtract the log of the model's own predicted marginal, add a uniform prior. Standard fix for a
# classifier whose test-time output distribution is skewed away from the true class balance.
if USABLE:
    for tag, r in USABLE.items():
        lg = r["logits"][:N_TEST]
        marg = softmax(lg).mean(0)                      # model's predicted marginal
        adj = np.log(np.clip(marg, 1e-12, None)) - np.log(1.0/20)
        corrected = lg - adj[None, :]
        changed = int((corrected.argmax(1) != lg.argmax(1)).sum())
        print(f"{tag}: prior correction changes {changed}/{N_TEST} predictions "
              f"({100*changed/N_TEST:.1f}%)")
        newdist = pd.Series(corrected.argmax(1)).value_counts(normalize=True)
        print(f"     max-class share {100*newdist.iloc[0]:.1f}% "
              f"(was {100*pd.Series(lg.argmax(1)).value_counts(normalize=True).iloc[0]:.1f}%)")
        write_submission(corrected, f"{tag}_priorcorrected")
        print(f"     NOTE: only submit this if §5 showed the skew is real. Prior correction\n"
              f"     assumes a uniform test prior, and {changed} clips is "
              f"{'more' if changed > 30 else 'less'} than the ~30 needed to move the score.")

In [ ]:
# --- remedy B: v3 + v4 ensemble -----------------------------------------------
if len(USABLE) >= 2:
    tags = list(USABLE)[:2]
    n = min(len(USABLE[t]["logits"]) for t in tags)
    ls = [np.log(np.clip(softmax(USABLE[t]["logits"][:n]), 1e-12, None)) for t in tags]
    ens = np.mean(ls, axis=0)
    for t in tags:
        ch = int((ens.argmax(1) != USABLE[t]["pred"][:n]).sum())
        print(f"ensemble differs from {t} on {ch}/{n} clips ({100*ch/n:.1f}%)")
    write_submission(ens, "ensemble_v3_v4")
    print("\n  An ensemble of two ~equal models usually lands at or slightly above the better one.\n"
          "  With a 35-clip gap to first place, this is the cheapest shot you have.")
else:
    print("Need both submissions loaded for the ensemble.")

In [ ]:
# --- remedy C: short-clip TTA, validated on labeled data first ------------------
if not (HAVE_GPU and TEST_OK and globals().get("_val") is not None):
    print("Skipping: needs a GPU, the test durations from §2, and ADI20 val from §3.")
else:
    SHORT_S = 3.0

    @torch.no_grad()
    def evaluate_tta(ds, n_windows=3, win_s=8.0, only_shorter_than=None, bs=16, desc=""):
        """Average log-softmax over n_windows evenly-spaced windows per clip."""
        P, Y = [], []
        for i in range(len(ds)):
            s = ds[i]
            try:
                arr, sr = s["audio"]["array"], int(s["audio"]["sampling_rate"])
            except Exception:
                continue
            w = torch.as_tensor(arr, dtype=torch.float32)
            if w.ndim > 1: w = w.mean(0)
            dur = len(w)/sr
            if only_shorter_than is not None and dur >= only_shorter_than:
                continue
            lab = labels2id.get(s.get("dialect"))
            if lab is None or w.numel() < 400:
                continue
            n = int(win_s*TARGET_SR)
            starts = ([0] if len(w) <= n
                      else np.linspace(0, len(w)-n, n_windows).astype(int).tolist())
            wins = [w[st:st+n].numpy() for st in starts]
            feats, mask = _extract_features(fe, wins)
            with torch.autocast("cuda", dtype=torch.bfloat16):
                lg = model(input_features=feats.to(device),
                           attention_mask=mask.to(device) if mask is not None else None)
            lp = torch.log_softmax(lg.float(), -1).mean(0)
            P.append(int(lp.argmax())); Y.append(lab)
        acc = 100*np.mean(np.array(P) == np.array(Y)) if P else float("nan")
        print(f"  {desc}: {acc:.2f}%  (n={len(P)})")
        return acc

    print(f"TTA on short clips only (< {SHORT_S}s), ADI20 val")
    a_1 = evaluate_tta(_val, n_windows=1, only_shorter_than=SHORT_S, desc="1 window (baseline)")
    a_3 = evaluate_tta(_val, n_windows=3, only_shorter_than=SHORT_S, desc="3 windows")
    print(f"\n  TTA gain on short clips: {a_3-a_1:+.2f} pts")
    share = float((test_dur < SHORT_S).mean())
    print(f"  short clips are {100*share:.1f}% of the test set, so a {a_3-a_1:+.2f} pt gain there\n"
          f"  is worth {(a_3-a_1)*share:+.2f} pts overall = {round((a_3-a_1)*share/100*N_TEST):+d} clips.")
    print(f"  {'WORTH SUBMITTING' if abs((a_3-a_1)*share/100*N_TEST) > 15 else 'below the noise floor -- not worth a submission slot'}")

## Summary

Fill this in as you go — the point of the notebook is to end with a ranked answer, not a folder
of plots.

| # | hypothesis | evidence | verdict |
|---|---|---|---|
| 1 | submission misaligned / doubled | §1 | |
| 2 | short clips | §2 §4 §5 | |
| 3 | skewed class prior | §4 §6A | |
| 4 | domain shift (residual) | §5 | |

**Remember the resolution limit.** At n=878 the leaderboard cannot distinguish anything smaller
than about ±3.3 points. Do not spend a submission on a change worth fewer than ~30 clips.